# 13 · Quét α, đường đánh đổi đầu tiên (ticket #19)

Xếp lại ứng viên của item-kNN liền kề bằng điểm mix theo luật (w 0.4/0.4/0.2) với α = 0; 0,1; …; 1.
- Độ chính xác (HitRate@10, NDCG@10, MRR) trên **cả** val.
- Độ mượt trên chuỗi 10 bài sinh tham lam (#18) từ **2.000** phiên val cố định (`random_seed` 42).

**Input** (Add Input): `titeohihi/mixnext-splited`, `mixnext-candidates`.

**Output** (`/kaggle/working/out/`): `sweep_v1.csv`, `fig_tradeoff_v1.png`. Không đăng Dataset: dán `sweep_v1.csv` vào PR #19, rồi bảng và hình (vẽ lại bằng `--plot-only`) được commit vào `reports/`.

Bật **Internet** để clone repo. Chỉ cần CPU.

In [ ]:
# Bootstrap: lấy code từ GitHub rồi cài package src/ (pyproject.toml, #3).
# Đang làm trên nhánh a/b/c thì đổi BRANCH; khi ticket đã merge thì để "main".
BRANCH = "b"
REPO = "/kaggle/working/MusicRecSys"
!rm -rf {REPO} && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys {REPO}
!pip install -q --no-deps -e {REPO}   # cho các lệnh `python -m src…` chạy ở tiến trình mới, từ thư mục bất kỳ
import sys; sys.path.insert(0, REPO)   # kernel đang chạy chỉ đọc gói mới cài khi khởi động lại: dòng này mới làm `import src` chạy được

In [ ]:
from pathlib import Path
INPUT = Path("/kaggle/input")
DATA = sorted(INPUT.rglob("eval_targets.parquet"))[0].parent
CANDIDATES = sorted(INPUT.rglob("itemknn_val_run0.parquet"))[0]
OUT = Path("/kaggle/working/out")
print("dữ liệu:", DATA)
print("ứng viên:", CANDIDATES)

In [ ]:
from src.eval.sweep import main

table = main(["--data", str(DATA), "--candidates", str(CANDIDATES), "--model", "itemknn", "--out", str(OUT), "--tag", "v1"])

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(OUT / "fig_tradeoff_v1.png")))
print((OUT / "sweep_v1.csv").read_text(encoding="utf-8"))   # dán khối này vào PR #19

## Kết quả (chạy trên Kaggle, CPU, ngày 27/11/2026)

item-kNN liền kề + mix theo luật (w 0.4/0.4/0.2). Độ chính xác trên 9.212 phiên val; độ mượt trên 2.000 phiên × 10 cặp = 20.000 cặp.

| α | NDCG@10 | HitRate@10 | MRR | % cặp mix được | % tông tương thích | ΔBPM TB (%) | lệch energy |
|---|---:|---:|---:|---:|---:|---:|---:|
| 1,0 (chỉ gu) | 0,1297 | 0,1921 | 0,1165 | 6,2 | 23,5 | 14,7 | 0,155 |
| 0,9 | 0,1291 | 0,1904 | 0,1160 | 10,0 | 29,1 | 13,5 | 0,150 |
| 0,8 | 0,1260 | 0,1837 | 0,1141 | 10,4 | 29,8 | 13,5 | 0,149 |
| **0,7 (được chọn)** | **0,1208** | **0,1738** | **0,1107** | **12,3** | **32,2** | **12,9** | **0,148** |
| 0,6 | 0,1131 | 0,1619 | 0,1045 | 17,8 | 38,9 | 11,6 | 0,144 |
| 0,5 | 0,1010 | 0,1512 | 0,0922 | 30,4 | 53,7 | 9,1 | 0,140 |
| 0,3 | 0,0561 | 0,1052 | 0,0496 | 72,6 | 89,7 | 4,1 | 0,115 |
| 0,0 (chỉ mix) | 0,0207 | 0,0456 | 0,0220 | 91,4 | 99,7 | 3,2 | 0,101 |

- Ngưỡng NDCG@10 = 0,9 × 0,1297 = **0,1168**. α = 0,7 là α nhỏ nhất còn trên ngưỡng (NDCG giảm 6,9%); α = 0,6 giảm 12,8%, vượt ngưỡng.
- Ở α = 0,7, tỉ lệ cặp mix được gấp đôi so với chỉ theo gu (6,2% → 12,3%), tông tương thích 23,5% → 32,2%.
- Đường đánh đổi gãy ở khoảng α 0,3–0,6: độ mượt tăng nhanh nhưng NDCG rơi mạnh. Với α = 0, tỉ lệ cặp mix được đạt 91%, nhưng NDCG chỉ còn 16% so với α = 1.
- Bảng đầy đủ: `reports/sweep_v1.csv`. Hình: `reports/fig_tradeoff_v1.png`, vẽ lại từ CSV bằng `--plot-only`.
